<a href="https://colab.research.google.com/github/VETTOORLIJO/DS_VettoorLijo/blob/master/classification_ml_bc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

from sklearn.pipeline import Pipeline


SyntaxError: invalid syntax (2586849190.py, line 13)

**Load the breast cancer dataset from sklearn.**

In [ ]:
BC_datasets = load_breast_cancer()
print(BC_datasets.keys())

In [ ]:
print(BC_datasets.filename)

In [ ]:
print(BC_datasets.target)

In [ ]:
print(BC_datasets.feature_names)

In [ ]:
df = pd.DataFrame(BC_datasets.data,columns=BC_datasets.feature_names)
df["target"] = BC_datasets.target
df.head()

Preprocess the data to handle any missing values and perform necessary feature
scaling.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes.value_counts()

In [ ]:
df['target'].value_counts()

In [ ]:
(df==0).sum() # df.duplicated().sum()   df.isna().sum() #df.isnull().sum()

*df is not balanced + their are zeros in [mean concavity ,mean concave points ,worst concavity, worst concave points,mean concave points,mean concavity] are not big numbers so ignore it

In [ ]:
plt.figure(figsize=(20,8))
sns.boxplot(data=df.melt(var_name="Feature",value_name="Value"),x="Feature",y="Value")
plt.xticks(rotation=90)
plt.show()

In [ ]:
#IQR
q1=df.quantile(0.25)
q3=df.quantile(0.75)
IQR=q3-q1
within_boundries=(df>=q1-1.5*IQR) & (df<=q3+1.5*IQR)
clean_df=df[within_boundries.all(axis=1)]
plt.figure(figsize=(20,8))
sns.boxplot(data=clean_df.melt(var_name="Feature",value_name="Value"),x="Feature",y="Value")
plt.xticks(rotation=90)
plt.show()

In [ ]:
clean_df['target'].value_counts()

NameError: name 'clean_df' is not defined

In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(df.corr(),annot=True)
plt.show()

In [ ]:
corr_target = df.corr(numeric_only=True)['target'].sort_values()
print(corr_target)

Logistic Regression model

In [ ]:
# two data frame
#1)  df
#2) clean_df

In [ ]:
result = []
classresult =[]


In [ ]:
X= df.drop('target',axis=1)
y=df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
def scaler_data(X_train,X_test):
    scaler = StandardScaler()
    X_train_scaler = scaler.fit_transform(X_train)
    X_test_scaler = scaler.transform(X_test)
    return X_train_scaler,X_test_scaler

In [ ]:
def evaluate_model(y_test, y_pred, model_name):

    result.append({
        "model": model_name,
        "accuracy_score": round(accuracy_score(y_test, y_pred), 4),
        "precision_score": round(precision_score(y_test, y_pred), 4),
        "recall_score": round(recall_score(y_test, y_pred), 4),
        "f1_score": round(f1_score(y_test, y_pred), 4),
        "confusion_matrix": confusion_matrix(y_test, y_pred)
    })

    classresult.append({
        "model": model_name,
        "classification_report":
            classification_report(y_test, y_pred)
    })

    return (
        pd.DataFrame(result),
        pd.DataFrame(classresult)
    )


In [ ]:
#Logistic Regression
X_train_scaled, X_test_scaled = scaler_data(X_train, X_test)
logic_Reg_model = LogisticRegression()
logic_Reg_model.fit(X_train_scaled,y_train)


In [ ]:
y_pred = logic_Reg_model.predict(X_test_scaled)

In [ ]:
#accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
results_df, reports_df  = evaluate_model(y_test, y_pred,"logic_Reg_model with outlier ")
display(results_df)

without outlier LogisticRegression

In [ ]:
X= clean_df.drop('target',axis=1)
y=clean_df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
#Logistic Regression
X_train_scaled, X_test_scaled = scaler_data(X_train, X_test)
logic_Reg_model = LogisticRegression()
logic_Reg_model.fit(X_train_scaled,y_train)


NameError: name 'scaler_data' is not defined

In [ ]:
y_pred = logic_Reg_model.predict(X_test_scaled)
results_df, reports_df  = evaluate_model(y_test, y_pred,"logic_Reg_model with out outlier ")
display(results_df)

LogisticRegression with parameters

In [ ]:
X= clean_df.drop('target',axis=1)
y=clean_df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
#Logistic Regression
X_train_scaled, X_test_scaled = scaler_data(X_train, X_test)
logic_Reg_model = LogisticRegression(
                                    penalty='l2',
                                    C=1.0,
                                    solver='lbfgs',
                                    max_iter=100,
                                    random_state=42
                                    )
logic_Reg_model.fit(X_train_scaled,y_train)

In [ ]:
y_pred = logic_Reg_model.predict(X_test_scaled)
results_df, reports_df  = evaluate_model(y_test, y_pred,"log______model with out outlier +Parameters  ")
display(results_df)

Decision Tree Classifier
#1)  df
#2) clean_df
criterion = ["gini", "entropy", "log_loss"]
max_depth = [3, 5, 10, None]
min_samples_leaf = [1, 2, 5]

In [ ]:
# two data frame
#1)  df
#2) clean_df
#criterion = ["gini", "entropy", "log_loss"]
#max_depth = [3, 5, 10, None]
#min_samples_leaf = [1, 2, 5]

In [ ]:
X= df.drop('target',axis=1)
y=df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
D_tree_model = DecisionTreeClassifier(
                                      criterion="gini",
                                      max_depth=None,
                                      min_samples_split=2,
                                      min_samples_leaf=1,
                                     )
D_tree_model.fit(X_train,y_train)

In [ ]:
y_pred = D_tree_model.fit(X_train,y_train).predict(X_test)
results_df, reports_df  = evaluate_model(y_test, y_pred,"D_tree(gini) Classifier model with  outlier")
display(results_df)

In [ ]:
D_tree_model = DecisionTreeClassifier(
                                      criterion="entropy",
                                      max_depth=None,
                                      min_samples_split=2,
                                      min_samples_leaf=1,
                                     )
D_tree_model.fit(X_train,y_train)

In [ ]:
y_pred = D_tree_model.fit(X_train,y_train).predict(X_test)
results_df, reports_df  = evaluate_model(y_test, y_pred,"D_tree(entropy) Classifier model with  outlier")
display(results_df)

In [ ]:
D_tree_model = DecisionTreeClassifier(
                                      criterion="log_loss",
                                      max_depth=None,
                                      min_samples_split=2,
                                      min_samples_leaf=1,
                                     )
D_tree_model.fit(X_train,y_train)

In [ ]:
y_pred = D_tree_model.fit(X_train,y_train).predict(X_test)
results_df, reports_df  = evaluate_model(y_test, y_pred,"D_tree(log_loss) Classifier model with  outlier")
display(results_df)

without outlier

In [ ]:
X= clean_df.drop('target',axis=1)
y=clean_df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
D_tree_model =DecisionTreeClassifier(
                                      splitter="random",
                                      random_state=42
                                  )
D_tree_model.fit(X_train,y_train)

In [ ]:
y_pred = D_tree_model.fit(X_train,y_train).predict(X_test)
results_df, reports_df  = evaluate_model(y_test, y_pred,"D_tree Classifier model with out outlier")
display(results_df)

Random Forest Classifier

    * n_estimators=50, 100,500
    * criterion="gini",criterion="entropy",criterion="log_loss",
    * max_depth=5,
    * max_features="sqrt", max_features="log2",max_features=None,
    * bootstrap=True, bootstrap=False,


In [ ]:
# two data frame
#1)  df
#2) clean_df
#  n_estimators=300,
# max_depth = [3, 5, 10, None]
# min_samples_leaf = [1, 2, 5]

In [ ]:
X= df.drop('target',axis=1)
y=df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
R_Forest_model = RandomForestClassifier(
                                      n_estimators=300,
                                      min_samples_split=2,
                                      min_samples_leaf=1,
                                     )
R_Forest_model.fit(X_train,y_train)

In [ ]:
y_pred = R_Forest_model.fit(X_train,y_train).predict(X_test)
results_df, reports_df  = evaluate_model(y_test, y_pred,"R_Forest_Classifier model with outlier")
display(results_df)

In [ ]:
X= clean_df.drop('target',axis=1)
y=clean_df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
R_Forest_model = RandomForestClassifier(
                                      n_estimators=300,
                                      min_samples_split=2,
                                      min_samples_leaf=1,
                                     )
R_Forest_model.fit(X_train,y_train)

In [ ]:
y_pred = R_Forest_model.fit(X_train,y_train).predict(X_test)
results_df, reports_df  = evaluate_model(y_test, y_pred,"R_Forest_Classifier model with out outlier")
display(results_df)

Support Vector Machine (SVM),

* kernel="linear", "rbf","sigmoid", ["poly", degree=3]
* C = [0.1, 1, 10]
* gamma = ["scale", 0.01, 0.1, 1]



In [ ]:
# two data frame
#1)  df
#2) clean_df
# kernel="linear", "rbf","sigmoid", ["poly", degree=3]
# C = [0.1, 1, 10]
# gamma = ["scale", 0.01, 0.1, 1]

In [ ]:
X= df.drop('target',axis=1)
y=df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train_scaled, X_test_scaled = scaler_data(X_train, X_test)
svm_models  = SVC(kernel="linear",C=1.0,gamma="scale")
svm_models.fit(X_train_scaled,y_train)

In [ ]:
y_pred = svm_models.predict(X_test_scaled)
results_df, reports_df  = evaluate_model(y_test, y_pred,"svm_models (linear) with outlier ")
display(results_df)

In [ ]:
X_train_scaled, X_test_scaled = scaler_data(X_train, X_test)
svm_models  = SVC(kernel="rbf",C=1.0,gamma="scale")
svm_models.fit(X_train_scaled,y_train)

In [ ]:
y_pred = svm_models.predict(X_test_scaled)
results_df, reports_df  = evaluate_model(y_test, y_pred,"svm_models (rbf) with outlier ")
display(results_df)

In [ ]:
X= clean_df.drop('target',axis=1)
y=clean_df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train_scaled, X_test_scaled = scaler_data(X_train, X_test)
svm_models  = SVC(kernel="sigmoid",C=1.0,gamma="scale")
svm_models.fit(X_train_scaled,y_train)

In [ ]:
y_pred = svm_models.predict(X_test_scaled)
results_df, reports_df  = evaluate_model(y_test, y_pred,"svm_models (linear) with out outlier ")
display(results_df)

K-Nearest Neighbors (KNN)

* 'n_neighbors': [3, 5, 7, 9],
*  weights': ['uniform', 'distance'],
*  metric': ['euclidean', 'manhattan'],
*    ['algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']]
*  cv=5,
* scoring='f1',
* n_jobs=-1


In [ ]:
# two data frame
#1)  df
# #2) clean_df
#     'n_neighbors': [3, 5, 7, 9],
#     'weights': ['uniform', 'distance'],
#     'metric': ['euclidean', 'manhattan'],
#     'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
#     *cv=5,
#     scoring='f1',
#     n_jobs=-1


In [ ]:
X= df.drop('target',axis=1)
y=df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train_scaled, X_test_scaled = scaler_data(X_train, X_test)

In [ ]:
#  hyperparameter tuning
param_grid = {
    'n_neighbors':[3,5,7,9,11],
    'weights':['uniform','distance'],
    'metric':['euclidean','manhattan']
}

In [ ]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(
                    KNeighborsClassifier(),
                    param_grid=param_grid,
                    cv=5,                              #Split training data into 5 parts ,Train on 4 parts, test on 1 part
                    scoring='f1',                       #Pick the parameter combination with the highest F1 score
                    n_jobs=-1                           #  n_jobs=-1  Use all CPU cores,  n_jobs=4  Uses four cores.
                  )
grid.fit(X_train_scaled, y_train)

In [ ]:
print(grid.best_params_)
print(grid.best_score_)

In [ ]:
best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test_scaled)
results_df, reports_df  = evaluate_model(y_test,y_pred,"Best KNN with outlier")
display(results_df)

In [ ]:
reports_df